In [32]:
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np
import scipy
import scipy.signal as sig
import cv2
import copy

In [33]:
colors = ("blue", "green", "red")
%matplotlib inline

In [ ]:
#inserire il percorso al .csv di interesse
raw_data = pd.read_csv("segnale.csv", index_col=0)
#selezionare i frame "buoni"
raw_data = raw_data.iloc[:,:]
raw_data.head()
#inserire il percorso del video per il calcolo del fr
video_path = "video.mp4"

In [35]:
def get_video_fps(video_path):
    # Apri il video
    cap = cv2.VideoCapture(video_path)

    # Controlla se il video è stato aperto correttamente
    if not cap.isOpened():
        print("Errore: impossibile aprire il video.")
        return None

    # Leggi il frame rate
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Frame rate del video: {fps:.2f} FPS")

    # Rilascia la risorsa
    cap.release()
    return fps

fr = get_video_fps(video_path)

fr = int(round(fr))
print(fr)

Frame rate del video: 30.00 FPS
30


In [36]:
t = np.arange(len(raw_data.iloc[:, 0].values)) / fr


In [37]:


fsizea = 30
fsizeb = 40
x_label = "time"
y_label = "avg_intensity"#settare avg_intensity/n_pixel in base a come si è ottenuto il sgn grezzo

def limit_signal(signal, threshold_high=None, threshold_low=None):
    """
    Limita i valori del segnale che superano le soglie specificate.
    Se threshold_high/low non sono specificati, usa 2 * std dev.
    """
    if threshold_high is None:
        threshold_high = np.mean(signal) + 2 * np.std(signal)
    if threshold_low is None:
        threshold_low = np.mean(signal) - 2 * np.std(signal)

    limited_signal = signal.copy()
    for i in range(len(limited_signal)):
        if limited_signal[i] > threshold_high:
            limited_signal[i] = threshold_high
        elif limited_signal[i] < threshold_low:
            limited_signal[i] = threshold_low
    return limited_signal

limited_sgn = raw_data.copy()

for i, col in enumerate(colors):
    # Estrai la colonna come array NumPy
    signal_array = raw_data.iloc[:, i].values

    # Applica la limitazione del segnale
    limited_array = limit_signal(signal_array)
    signal_min = np.min(limited_array)
    signal_max = np.max(limited_array) 
    normalized_signal = -1 + ((limited_array - signal_min ) * (2)) / (signal_max - signal_min)


    # Salva il segnale limitato
    limited_sgn.iloc[:, i] = normalized_signal


In [38]:
chosen_sgn = copy.deepcopy(limited_sgn)


fixed_freq = np.zeros(3)#qui saranno salvate le stime HR1, utili per aggiustare i parametri
i_peaks = []
RR_intervals_0 = []
for i, color in enumerate(colors):
    peaks,_ = sig.find_peaks(chosen_sgn.iloc[:,i], height=-0.25, distance=0.39*fr,threshold=None, prominence=None, width=None, wlen=None,  plateau_size=None)
  
    intervalli = []
    for j in range(0, len(list(peaks))-1):
        intervalli.append(list(peaks)[j+1]-list(peaks)[j])
    i_peaks.append(list(peaks))
    RR_intervals_0.append(intervalli)
    fixed_freq[i] = len(peaks)/(len(chosen_sgn.iloc[:,i])-1)*fr*60
    #(numero di picchi/numero di frame) * frame al secondo = frequenza in Hz, *60 = al minuto
    print(f"frequenza cardiaca calcolata usando il canale {color} = {fixed_freq[i]}")



frequenza cardiaca calcolata usando il canale blue = 77.42479617655327
frequenza cardiaca calcolata usando il canale green = 83.49732921000843
frequenza cardiaca calcolata usando il canale red = 76.91875175709868


Il blocco qui sotto restituisce la migliore stima per il canale del rosso stimando la distanza tra picchi minima ottimale prima con una retta a m negativo e poi con una iperbole 

In [39]:
d01 = 0.43
md = 0.004
h1 = -0.4
d02 = 0.037
ad = 25
h2 = -0.2
freq1 = np.zeros(3)
freq2 = np.zeros(3)
d1 = d01 - (fixed_freq[2]-70)*md #metodo retta
d2 = d02 + ad/(fixed_freq[2]) #metodo iperbole
#print(f"{d1} {h1}\n{d2} {h2}")
i_peaks1 = []
i_peaks2 = []
RR_intervals_1 = []
RR_intervals_2 = []
for i, color in enumerate(colors):
    d1 = d01 - (fixed_freq[i]-70)*md
    d2 = d02 + ad/(fixed_freq[i])

    #metodo retta
    peaks,_ = sig.find_peaks(chosen_sgn.iloc[:,i], height=h1, distance=d1*fr,threshold=None, prominence=None, width=None, wlen=None,  plateau_size=None)
   # print(i_peaks)
    
    i_peaks1.append(list(peaks))
    freq1[i] = len(peaks)/(len(chosen_sgn.iloc[:,i])-1)*fr*60
    intervalli = []
    for j in range(0, len(list(peaks))-1):
        intervalli.append(list(peaks)[j+1]-list(peaks)[j])
    RR_intervals_1.append(intervalli)

    #metodo iperbole
    peaks,_ = sig.find_peaks(chosen_sgn.iloc[:,i], height=h2, distance=d2*fr,threshold=None, prominence=None, width=None, wlen=None,  plateau_size=None)
   
    intervalli = []
    for j in range(0, len(list(peaks))-1):
        intervalli.append(list(peaks)[j+1]-list(peaks)[j])
    RR_intervals_2.append(intervalli)
    i_peaks2.append(list(peaks))
    freq2[i] = len(peaks)/(len(chosen_sgn.iloc[:,i])-1)*fr*60
    #(numero di picchi/numero di frame) * frame al secondo = frequenza in Hz, *60 = al minuto
print("\t\tblu\tverde\trosso")
print(f"HR1\t\t{round(fixed_freq[0],1)}\t{round(fixed_freq[1],1)}\t{round(fixed_freq[2],1)}".replace('.', ','))
print(f"HR2 retta\t{round(freq1[0],1)}\t{round(freq1[1],1)}\t{round(freq1[2],1)}".replace('.', ','))
print(f"HR2 iperbole\t{round(freq2[0],1)}\t{round(freq2[1],1)}\t{round(freq2[2],1)}".replace('.', ','))

		blu	verde	rosso
HR1		77,4	83,5	76,9
HR2 retta	75,9	87,0	74,9
HR2 iperbole	83,5	87,0	79,4


Ora si selezionano, con opportuni criteri, le finestre temporali in cui il segnale è regolare

In [40]:
def compute_stats(i_peaks):
    '''
    riceve l'indice dei frame con picchi
    calcola e restituisce mediana e percentili 10,90
    '''
    median = np.zeros(3)
    p10 = np.zeros(3)
    p90 = np.zeros(3)

    for i in range(3):
        peaks = i_peaks[i]

        if len(peaks) < 2:
            continue

        intervals = [peaks[j+1] - peaks[j] for j in range(len(peaks)-1)]

        median[i] = np.median(intervals)
        p10[i] = np.percentile(intervals, 10)
        p90[i] = np.percentile(intervals, 90)

    return median, p10, p90

In [41]:
def clean_peaks(peaks, median_i, p10_i, p90_i, alpha_1, alpha_2):
    '''
    Riceve: indice frame picchi, mediana e percentili, valori di alpha da usare per scartare i picchi
    Restituisce: i nuovi indici dei picchi, i picchi rimossi e gli intervalli corrispondenti
    '''
    keep = [True] * len(peaks)
    intervals = []

    j = 1
    while j < len(peaks) - 1:

        prev_diff = peaks[j] - peaks[j-1]
        next_diff = peaks[j+1] - peaks[j]
        #primi due criteri
        cond1 = (
            prev_diff < alpha_2 * median_i or
            prev_diff > alpha_1 * median_i
        )
        #seconda coppia di criteri
        cond2 = (
            (prev_diff < p10_i and next_diff > p90_i) or
            (next_diff < p10_i and prev_diff > p90_i)
        )

        if cond1 or cond2:
            # intervallo da rimuovere
            start = peaks[j-1]
            end = peaks[j+1]

            intervals.append((start, end))

            # rimuovi picchi
            keep[j] = False
            keep[j+1] = False

            j += 2
        else:
            j += 1

    new_peaks = [p for p, k in zip(peaks, keep) if k]
    deleted = [p for p, k in zip(peaks, keep) if not k]

    return new_peaks, deleted, intervals

In [42]:
#Rimossi gli intervalli irregolari, verifica eventuali intervalli scartati consecutivi
def merge_intervals(intervals):
    if not intervals:
        return []

    intervals.sort()
    merged = [list(intervals[0])]

    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    return merged

In [43]:
def process_peak_set(i_peaks, alpha_1, alpha_2):
    '''
    Coordina le funzioni precedenti per procedere alla selezione delle finestre temporali
    '''
    median, p10, p90 = compute_stats(i_peaks)

    new_i_peaks = []
    deleted_peaks = []
    i_intervals = []
    counters1 = [0,0,0]
    counters2 = [0,0,0]

    for i in range(3):
        new_peaks, deleted, intervals = clean_peaks(
            i_peaks[i],
            median[i],
            p10[i],
            p90[i],
            alpha_1,
            alpha_2
        )

        intervals = merge_intervals(intervals)

        new_i_peaks.append(new_peaks)
        deleted_peaks.append(deleted)
        if intervals == []:
            i_intervals.append([[0, 0]])
        else:
            i_intervals.append(intervals)
    return new_i_peaks, deleted_peaks, i_intervals, median, p10, p90

In [44]:
# taglio diretto
original_peaks0 = copy.deepcopy(i_peaks)
new_i_peaks0, deleted0, intervals0, median0, p100, p900 = process_peak_set(i_peaks, 2.3, 0.4)
freq_out0 = np.zeros(3)
for i, col in enumerate(colors):
    removed = sum(end - start for start, end in intervals0[i])
    effective_len = len(chosen_sgn.iloc[:, i]) - removed
    freq_out0[i] = len(new_i_peaks0[i]) / (effective_len-1)* fr * 60

d01 = 0.445
md = 0.0052
h1 = -0.2
d02 = 0.045
ad = 28
h2 = -0.2
freq1 = np.zeros(3)
freq2 = np.zeros(3)
d1 = d01 - (fixed_freq[2]-70)*md
d2 = d02 + ad/(fixed_freq[2])

i_peaks1 = []#retta+taglio
i_peaks2 = []#iperbole+taglio
for i, color in enumerate(colors):
    d1 = d01 - (fixed_freq[i]-70)*md
    d2 = d02 + ad/(fixed_freq[i])
#taglio diretto
    peaks,_ = sig.find_peaks(chosen_sgn.iloc[:,i], height=h1, distance=d1*fr,threshold=None, prominence=None, width=None, wlen=None,  plateau_size=None)
   
    i_peaks1.append(list(peaks))
    freq1[i] = len(peaks)/(len(chosen_sgn.iloc[:,i])-1)*fr*60

    peaks,_ = sig.find_peaks(chosen_sgn.iloc[:,i], height=h2, distance=d2*fr,threshold=None, prominence=None, width=None, wlen=None,  plateau_size=None)
 
    i_peaks2.append(list(peaks))
    freq2[i] = len(peaks)/(len(chosen_sgn.iloc[:,i])-1)*fr*60
    #(numero di picchi/numero di frame) * frame al secondo = frequenza in Hz, *60 = al minuto

# taglio+retta
original_peaks1 = copy.deepcopy(i_peaks1)
new_i_peaks1, deleted1, intervals1, median1, p101, p901 = process_peak_set(i_peaks1, 1.7, 0.6)
freq_out1 = np.zeros(3)
for i, col in enumerate(colors):
    removed = sum(end - start for start, end in intervals1[i])
    effective_len = len(chosen_sgn.iloc[:, i]) - removed
    freq_out1[i] = len(new_i_peaks1[i]) / (effective_len-1)* fr * 60

# taglio+iperbole
original_peaks2 = copy.deepcopy(i_peaks2)
new_i_peaks2, deleted2, intervals2, median2, p102, p902 = process_peak_set(i_peaks2, 1.7, 0.6)
freq_out2 = np.zeros(3)
for i, col in enumerate(colors):
    removed = sum(end - start for start, end in intervals2[i])
    effective_len = len(chosen_sgn.iloc[:, i]) - removed
    freq_out2[i] = len(new_i_peaks2[i]) / (effective_len-1)* fr * 60
print("\t\tblu\tverde\trosso")
print(f"HR3 fixed\t{round(freq_out0[0],1)}\t{round(freq_out0[1],1)}\t{round(freq_out0[2],1)}".replace('.', ','))
print(f"HR3 retta\t{round(freq_out1[0],1)}\t{round(freq_out1[1],1)}\t{round(freq_out1[2],1)}".replace('.', ','))
print(f"HR3 iperbole\t{round(freq_out2[0],1)}\t{round(freq_out2[1],1)}\t{round(freq_out2[2],1)}".replace('.', ','))

		blu	verde	rosso
HR3 fixed	77,0	87,2	75,2
HR3 retta	78,4	83,3	78,6
HR3 iperbole	78,4	83,3	78,6


In [45]:
print(new_i_peaks0)
print(deleted0)
print(intervals0)
RR_intervals_0T = []
for i, col in enumerate(colors):
    intervalli = []
    partenze_intervalli_scartati = []
    fine_intervalli_scartati = []
    for intervallo in intervals0[i]:
        partenza = intervallo[0]
        arrivo = intervallo[-1]
        partenze_intervalli_scartati.append(partenza)
        fine_intervalli_scartati.append(arrivo)
    
    for j in range(len(new_i_peaks0[i]) - 1):
        start = new_i_peaks0[i][j]
        next_peak = new_i_peaks0[i][j+1]
        
        if start in partenze_intervalli_scartati:
            idx = partenze_intervalli_scartati.index(start)
            end = fine_intervalli_scartati[idx]
            intervalli.append(next_peak - end)
        else:
            intervalli.append(next_peak - start)
    
    RR_intervals_0T.append(intervalli)


RR_intervals_1T = []
for i, col in enumerate(colors):
    intervalli = []
    partenze_intervalli_scartati = []
    fine_intervalli_scartati = []
    for intervallo in intervals1[i]:
        partenza = intervallo[0]
        arrivo = intervallo[-1]
        partenze_intervalli_scartati.append(partenza)
        fine_intervalli_scartati.append(arrivo)
    
    for j in range(len(new_i_peaks1[i]) - 1):
        start = new_i_peaks1[i][j]
        next_peak = new_i_peaks1[i][j+1]
        
        if start in partenze_intervalli_scartati:
            idx = partenze_intervalli_scartati.index(start)
            end = fine_intervalli_scartati[idx]
            intervalli.append(next_peak - end)
        else:
            intervalli.append(next_peak - start)
    
    RR_intervals_1T.append(intervalli)


RR_intervals_2T = []
for i, col in enumerate(colors):
    intervalli = []
    partenze_intervalli_scartati = []
    fine_intervalli_scartati = []
    for intervallo in intervals2[i]:
        partenza = intervallo[0]
        arrivo = intervallo[-1]
        partenze_intervalli_scartati.append(partenza)
        fine_intervalli_scartati.append(arrivo)
    
    for j in range(len(new_i_peaks2[i]) - 1):
        start = new_i_peaks2[i][j]
        next_peak = new_i_peaks2[i][j+1]
        
        if start in partenze_intervalli_scartati:
            idx = partenze_intervalli_scartati.index(start)
            end = fine_intervalli_scartati[idx]
            intervalli.append(next_peak - end)
        else:
            intervalli.append(next_peak - start)
    
    RR_intervals_2T.append(intervalli)

[[np.int64(15), np.int64(66), np.int64(91), np.int64(198), np.int64(211), np.int64(237), np.int64(252), np.int64(265), np.int64(284), np.int64(302), np.int64(331), np.int64(362), np.int64(389), np.int64(413), np.int64(438), np.int64(466), np.int64(485), np.int64(512), np.int64(537), np.int64(557), np.int64(571), np.int64(602), np.int64(626), np.int64(658), np.int64(693), np.int64(717), np.int64(737), np.int64(760), np.int64(799), np.int64(844), np.int64(878), np.int64(921), np.int64(941), np.int64(964), np.int64(989), np.int64(1013), np.int64(1033), np.int64(1065), np.int64(1078), np.int64(1102), np.int64(1122), np.int64(1144), np.int64(1166), np.int64(1191), np.int64(1210), np.int64(1233), np.int64(1257), np.int64(1279), np.int64(1302), np.int64(1323), np.int64(1346), np.int64(1365), np.int64(1390), np.int64(1415), np.int64(1437), np.int64(1454), np.int64(1467), np.int64(1481), np.int64(1503), np.int64(1527), np.int64(1544), np.int64(1571), np.int64(1604), np.int64(1617), np.int64(163

In [46]:
#calcolo HRV rmssd
rmssd_0 = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_0[i])-1):
        somma += ((RR_intervals_0[i][j+1]-RR_intervals_0[i][j])/fr)**2
    rmssd_0.append(np.sqrt(somma/(len(RR_intervals_0[i])-1)))
rmssd_1 = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_1[i])-1):
        somma += ((RR_intervals_1[i][j+1]-RR_intervals_1[i][j])/fr)**2
    rmssd_1.append(np.sqrt(somma/(len(RR_intervals_1[i])-1)))
rmssd_2 = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_2[i])-1):
        somma += ((RR_intervals_2[i][j+1]-RR_intervals_2[i][j])/fr)**2
    rmssd_2.append(np.sqrt(somma/(len(RR_intervals_2[i])-1)))
rmssd_0T = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_0T[i])-1):
        somma += ((RR_intervals_0T[i][j+1]-RR_intervals_0T[i][j])/fr)**2
    rmssd_0T.append(np.sqrt(somma/(len(RR_intervals_0T[i])-1)))
rmssd_1T = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_1T[i])-1):
        somma += ((RR_intervals_1T[i][j+1]-RR_intervals_1T[i][j])/fr)**2
    rmssd_1T.append(np.sqrt(somma/(len(RR_intervals_1T[i])-1)))
rmssd_2T = []
for i, col in enumerate(colors):
    somma = 0
    for j in range(0, len(RR_intervals_2T[i])-1):
        somma += ((RR_intervals_2T[i][j+1]-RR_intervals_2T[i][j])/fr)**2
    rmssd_2T.append(np.sqrt(somma/(len(RR_intervals_2T[i])-1)))
print("HRV in secondi, si nota che, indipendentemente dalla stima usata i valori sono fuori dal range fisiologico")
print(f"{rmssd_0[0]}\t{rmssd_0[1]}\t{rmssd_0[2]}\t{rmssd_0T[0]}\t{rmssd_0T[1]}\t{rmssd_0T[2]}")
print(f"{rmssd_1[0]}\t{rmssd_1[1]}\t{rmssd_1[2]}\t{rmssd_1T[0]}\t{rmssd_1T[1]}\t{rmssd_1T[2]}")
print(f"{rmssd_2[0]}\t{rmssd_2[1]}\t{rmssd_2[2]}\t{rmssd_2T[0]}\t{rmssd_2T[1]}\t{rmssd_2T[2]}")


HRV in secondi, si nota che, indipendentemente dalla stima usata i valori sono fuori dal range fisiologico
0.399254271749137	0.336882940919771	0.2951333659613967	0.3284245032286531	0.3124234323542503	0.24123524710216876
0.3667178506454002	0.3212211178237629	0.2933172473796079	0.2768312013099165	0.29251412979381397	0.17515425300482715
0.4034108157927457	0.3552132747916881	0.30659419433511786	0.2768312013099165	0.29251412979381397	0.17515425300482715
